<a href="https://colab.research.google.com/github/HillaryDrugs/li7/blob/main/CNN_%2B_CTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install torch torchaudio datasets jiwer numpy soundfile huggingface_hub matplotlib

import io
import re
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import soundfile as sf
import matplotlib.pyplot as plt

from datasets import load_dataset, Audio
from jiwer import process_words
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader

# -----------------------------
# 0) Setup
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

REPO_ID = "NightPrince/MasriSpeech-Full"
TARGET_SR = 16000

# -----------------------------
# 1) Load STREAMING dataset (audio decode disabled to avoid torchcodec)
# -----------------------------
ds_stream = load_dataset(REPO_ID, streaming=True)
ds_stream = ds_stream.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=False))
train_stream = ds_stream["train"]
val_stream   = ds_stream["validation"]

# -----------------------------
# 2) Text normalization
# -----------------------------
def normalize_ar(text: str) -> str:
    text = (text or "").strip()
    # remove tashkeel
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)
    # normalize alef variants
    text = re.sub("[إأآا]", "ا", text)
    # normalize ya + taa marbuta
    text = text.replace("ى", "ي").replace("ة", "ه")
    # remove tatweel
    text = text.replace("ـ", "")
    # keep Arabic + spaces
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

# -----------------------------
# 3) WER breakdown (format like your screenshot)
# -----------------------------
def compute_wer_breakdown(preds, refs):
    """
    Prints exactly:
      Perfect WER: x.x   (1 - exact match rate)
      Deletion WER: x.x  (D/N)
      Typo WER: x.x      (S/N)  substitution rate
    """
    preds = [normalize_ar(p) for p in preds]
    refs  = [normalize_ar(r) for r in refs]

    total_words = 0
    total_del = 0
    total_sub = 0
    perfect = 0

    for r, h in zip(refs, preds):
        m = process_words(r, h)
        N = m.hits + m.substitutions + m.deletions
        total_words += N
        total_del += m.deletions
        total_sub += m.substitutions
        if (m.substitutions + m.deletions + m.insertions) == 0:
            perfect += 1

    perfect_wer  = 1 - (perfect / len(refs)) if refs else 0.0
    deletion_wer = (total_del / total_words) if total_words else 0.0
    typo_wer     = (total_sub / total_words) if total_words else 0.0

    print(f"Perfect WER: {perfect_wer:.1f}")
    print(f"Deletion WER: {deletion_wer:.1f}")
    print(f"Typo WER: {typo_wer:.1f}")

    return perfect_wer, deletion_wer, typo_wer

# -----------------------------
# 4) Build character vocab from TRAIN transcriptions only
# -----------------------------
def build_vocab_from_stream(stream, max_items=20000):
    chars = set()
    for i, ex in enumerate(stream):
        t = normalize_ar(ex["transcription"])
        chars.update(list(t))
        if i + 1 >= max_items:
            break
    chars = sorted(list(chars))
    if " " not in chars:
        chars.append(" ")
    id2ch = ["<blank>"] + chars
    ch2id = {c:i for i,c in enumerate(id2ch)}
    return ch2id, id2ch

train_stream_for_vocab = load_dataset(REPO_ID, streaming=True)
train_stream_for_vocab = train_stream_for_vocab.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=False))["train"]

ch2id, id2ch = build_vocab_from_stream(train_stream_for_vocab, max_items=20000)
blank_id = 0
vocab_size = len(id2ch)
print("Vocab size:", vocab_size)

def text_to_ids(text: str):
    t = normalize_ar(text)
    return [ch2id[c] for c in t if c in ch2id]

# -----------------------------
# 5) Audio loader (download file by repo-relative path)
# -----------------------------
resamplers = {}

def load_audio_16k(audio_obj):
    if isinstance(audio_obj, dict) and audio_obj.get("bytes", None):
        wav, sr = sf.read(io.BytesIO(audio_obj["bytes"]))
    else:
        fname = audio_obj["path"]
        local_path = hf_hub_download(repo_id=REPO_ID, filename=fname, repo_type="dataset")
        wav, sr = sf.read(local_path)

    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)

    if sr != TARGET_SR:
        if sr not in resamplers:
            resamplers[sr] = torchaudio.transforms.Resample(sr, TARGET_SR)
        wav_t = torch.tensor(wav).unsqueeze(0)
        wav = resamplers[sr](wav_t).squeeze(0).numpy().astype(np.float32)

    return wav

# -----------------------------
# 6) Log-Mel Features
# -----------------------------
n_mels = 80
mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR, n_fft=400, hop_length=160, n_mels=n_mels
).to(device)
to_db = torchaudio.transforms.AmplitudeToDB().to(device)

@torch.no_grad()
def wav_to_logmel(wav_np: np.ndarray) -> torch.Tensor:
    x = torch.tensor(wav_np, dtype=torch.float32, device=device).unsqueeze(0)
    m = mel(x)
    m = to_db(m).clamp(min=-80.0)
    m = (m + 80.0) / 80.0
    return m.squeeze(0).transpose(0, 1)  # (frames, 80)

# -----------------------------
# 7) Iterable dataset wrapper + collate
# -----------------------------
class StreamWrapper(torch.utils.data.IterableDataset):
    def __init__(self, stream, limit=None):
        self.stream = stream
        self.limit = limit

    def __iter__(self):
        for i, ex in enumerate(self.stream):
            yield ex
            if self.limit is not None and (i + 1) >= self.limit:
                break

def collate_fn(batch):
    feats, feat_lens = [], []
    labels, label_lens = [], []
    refs = []

    for ex in batch:
        wav = load_audio_16k(ex["audio"])
        txt = ex["transcription"]

        f = wav_to_logmel(wav)
        y = text_to_ids(txt)

        feats.append(f)
        feat_lens.append(f.shape[0])
        labels.append(torch.tensor(y, dtype=torch.long))
        label_lens.append(len(y))
        refs.append(txt)

    maxT = max(feat_lens)
    B = len(batch)
    feat_pad = torch.zeros(B, maxT, n_mels, device=device)
    for i, f in enumerate(feats):
        feat_pad[i, :f.shape[0]] = f

    label_cat = torch.cat(labels) if labels else torch.empty(0, dtype=torch.long)
    return (
        feat_pad,
        torch.tensor(feat_lens, device=device, dtype=torch.long),
        label_cat.to(device),
        torch.tensor(label_lens, device=device, dtype=torch.long),
        refs
    )

TRAIN_LIMIT = None
VAL_LIMIT   = None

# recreate streams (iterators get consumed)
train_stream = load_dataset(REPO_ID, streaming=True)
train_stream = train_stream.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=False))["train"]
val_stream   = load_dataset(REPO_ID, streaming=True)
val_stream   = val_stream.cast_column("audio", Audio(sampling_rate=TARGET_SR, decode=False))["validation"]

train_loader = DataLoader(StreamWrapper(train_stream, limit=TRAIN_LIMIT),
                          batch_size=4, collate_fn=collate_fn)
val_loader   = DataLoader(StreamWrapper(val_stream, limit=VAL_LIMIT),
                          batch_size=4, collate_fn=collate_fn)

print("Training limit:", TRAIN_LIMIT)
print("Validation: FULL (streaming)")

# -----------------------------
# 8) CNN + CTC Model
# -----------------------------
def conv1d_out_len(L, kernel, stride, pad, dilation=1):
    return (L + 2*pad - dilation*(kernel-1) - 1) // stride + 1

class CNN_CTC(nn.Module):
    def __init__(self, n_mels, vocab_size):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 256, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(256, vocab_size)
        self._layers = [(5,2,2),(5,2,2),(3,1,1)]

    def forward(self, x, x_lens):
        x = x.transpose(1,2)      # (B,80,T)
        h = self.conv(x)          # (B,C,T')
        h = h.transpose(1,2)      # (B,T',C)
        logits = self.classifier(h)

        out_lens = x_lens.clone()
        for (k,s,p) in self._layers:
            out_lens = torch.tensor(
                [conv1d_out_len(int(L), k, s, p) for L in out_lens.tolist()],
                device=x_lens.device, dtype=torch.long
            )
        return logits, out_lens

model = CNN_CTC(n_mels=n_mels, vocab_size=vocab_size).to(device)
ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)

# -----------------------------
# 9) Manual CTC Decoding
# -----------------------------
def manual_ctc_decode(token_ids):
    out = []
    prev = None
    for t in token_ids:
        if t == prev:
            continue
        if t == blank_id:
            prev = t
            continue
        out.append(id2ch[t])
        prev = t
    return "".join(out)

# -----------------------------
# 10) Train + Evaluate FULL validation metrics
# -----------------------------
def train_one_epoch():
    model.train()
    total = 0.0
    steps = 0
    for feat, feat_lens, y, y_lens, _ in train_loader:
        opt.zero_grad()
        logits, out_lens = model(feat, feat_lens)
        log_probs = logits.log_softmax(dim=-1).permute(1,0,2)  # (T',B,V)
        loss = ctc_loss(log_probs, y, out_lens, y_lens)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.item())
        steps += 1
    return total / max(1, steps)

@torch.no_grad()
def eval_full_val_metrics():
    model.eval()
    preds, refs = [], []
    for feat, feat_lens, _, _, ref_texts in val_loader:
        logits, out_lens = model(feat, feat_lens)
        pred_ids = logits.argmax(dim=-1)
        for i in range(pred_ids.size(0)):
            seq = pred_ids[i, :int(out_lens[i])].tolist()
            preds.append(manual_ctc_decode(seq))
        refs.extend(ref_texts)

    # Prints exactly like the screenshot:
    return compute_wer_breakdown(preds, refs)

EPOCHS = 3
print("\nStarting CNN+CTC training on MasriSpeech-Full (Arabic only)...")
for ep in range(1, EPOCHS + 1):
    loss = train_one_epoch()
    print(f"\nEpoch {ep}/{EPOCHS} | Train loss: {loss:.4f}")
    eval_full_val_metrics()

print("\n✅ Done.")
